In [1]:
import torch
import torch.nn
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
from torch.utils.data import Dataset, DataLoader
import os

In [2]:

if os.path.exists('maliks_muwatta.csv'):
    df = pd.read_csv('maliks_muwatta.csv', encoding='utf-8')
else:
# الرابط الصحيح
    url = "https://raw.githubusercontent.com/abdelrahmaan/Hadith-Data-Sets/master/All%20Hadith%20Books/Maliks%20Muwatta%20Without_Tashkel.csv"

    # تحميل
    df = pd.read_csv(url, encoding='utf-8')

print(df.head())
print(df.shape)
print(df.columns)

# حفظ محليا
df.to_csv('maliks_muwatta.csv', index=False, encoding='utf-8')


                      Maliks Muwatta Without_Tashkel
0  قال حدثني الليثي عن مالك بن أنس عن ابن شهاب أن...
1  و حدثني يحيى عن مالك عن زيد بن أسلم عن عطاء بن...
2  و حدثني يحيى عن مالك عن يحيى بن سعيد عن عمرة ب...
3  و حدثني عن مالك عن زيد بن أسلم عن عطاء بن يسار...
4  و حدثني عن مالك عن نافع مولى عبد الله بن عمر أ...
(1594, 1)
Index(['Maliks Muwatta Without_Tashkel'], dtype='object')


In [3]:
texts = df['Maliks Muwatta Without_Tashkel'][:500].tolist()
print(texts[:5])  # أول 5 نصوص

['قال حدثني الليثي عن مالك بن أنس عن ابن شهاب أن عمر بن عبد العزيز أخر الصلاة يوما فدخل عليه عروة بن الزبير فأخبره أن المغيرة بن شعبة أخر الصلاة يوما وهو بالكوفة فدخل عليه أبو مسعود الأنصاري فقال ما هذا يا مغيرة أليس قد علمت أن جبريل نزل فصلى فصلى رسول الله صلى الله عليه وسلم ثم صلى فصلى رسول الله صلى الله عليه وسلم ثم صلى فصلى رسول الله صلى الله عليه وسلم ثم صلى فصلى رسول الله صلى الله عليه وسلم ثم صلى فصلى رسول الله صلى الله عليه وسلم ثم قال بهذا أمرت فقال عمر بن عبد العزيز اعلم ما تحدث به يا عروة أو إن جبريل هو الذي أقام لرسول الله صلى الله عليه وسلم وقت الصلاة قال عروة كذلك كان بشير بن أبي مسعود الأنصاري يحدث عن أبيه قال عروة ولقد حدثتني عائشة زوج النبي صلى الله عليه وسلم أن رسول الله صلى الله عليه وسلم كان يصلي العصر والشمس في حجرتها قبل أن تظهر.', 'و حدثني يحيى عن مالك عن زيد بن أسلم عن عطاء بن يسار أنه قال جاء رجل إلى رسول الله صلى الله عليه وسلم فسأله عن وقت صلاة الصبح قال فسكت عنه رسول الله صلى الله عليه وسلم حتى إذا كان من الغد صلى الصبح حين طلع الفجر ثم صلى الصبح من الغد بعد

In [ ]:
joined_text = '\n'.join(texts)

In [ ]:
del df,texts

In [6]:
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
tokens = word_tokenize(joined_text)
print(tokens[:10])  # أول 5 نصوص بعد التوكنيزشن

['قال', 'حدثني', 'الليثي', 'عن', 'مالك', 'بن', 'أنس', 'عن', 'ابن', 'شهاب']


In [ ]:
unique_tokens = np.unique(tokens)
print(f"Unique tokens len: {len(unique_tokens)}")

Unique tokens len: 4083


In [ ]:
word2idx = {word: idx for idx, word in enumerate(unique_tokens)}
idx2word = {idx: word for word, idx in word2idx.items()}

In [ ]:
n_words = 2 # Number of words to look at before predicting the next word
sequences = []
next_words = []

for i in range(len(tokens)-n_words):
    sequences.append(tokens[i:i+n_words])
    next_words.append(tokens[i+n_words])

In [11]:
X = np.zeros((len(sequences), n_words,len(unique_tokens)), dtype=bool)
y = np.zeros((len(next_words), len(unique_tokens)), dtype=bool)

for i, sequence in enumerate(sequences):
    for t, word in enumerate(sequence):
        X[i, t, word2idx[word]] = 1
    y[i, word2idx[next_words[i]]] = 1

In [ ]:
import torch.nn as nn

class LSTMTextGenerator(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2):
        super().__init__()
        self.embedding = nn.Linear(vocab_size, embedding_dim)  # or use nn.Embedding if using token indices
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=0.3
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)
        # softmax
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)  # (batch, seq_len, embedding_dim)
        lstm_out, _ = self.lstm(x)  # (batch, seq_len, hidden_dim)
        logits = self.fc(lstm_out)  # (batch, seq_len, vocab_size)
        logits = self.softmax(logits)
        return logits


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [14]:
from torch.utils.tensorboard import SummaryWriter
# Reinitialize with smaller embedding_dim
vocab_size = len(unique_tokens)
embedding_dim = 128  # Reduced from vocab_size
hidden_dim = 256
model = LSTMTextGenerator(vocab_size, embedding_dim, hidden_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)  # Reduced learning rate
writer = SummaryWriter()


In [16]:
del joined_text,sequences,next_words,word2idx,idx2word

In [17]:
dataset = torch.utils.data.TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

In [18]:
from torch.optim.lr_scheduler import CosineAnnealingLR
# train model
num_epochs = 100
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)


for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in dataloader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)  # (batch_size, seq_len, vocab_size)
        # Take only the last timestep: (batch_size, vocab_size)
        last_outputs = outputs[:, -1, :]
        loss = criterion(last_outputs, batch_y.argmax(dim=1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    
    # تسجيل LR
    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('Learning Rate', current_lr, epoch)
    writer.add_scalar('Loss', avg_loss, epoch)
    scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader)}")
writer.close()

Epoch 1/100, Loss: 5.624993958270042
Epoch 2/100, Loss: 4.763542275679739
Epoch 3/100, Loss: 4.459692387951346
Epoch 4/100, Loss: 4.195662543887184
Epoch 5/100, Loss: 3.927364692950906
Epoch 6/100, Loss: 3.667906051291559
Epoch 7/100, Loss: 3.424720745636407
Epoch 8/100, Loss: 3.201606482192687
Epoch 9/100, Loss: 2.9958710610717163
Epoch 10/100, Loss: 2.8118175949369157
Epoch 11/100, Loss: 2.6496793671060623
Epoch 12/100, Loss: 2.5031287580504453
Epoch 13/100, Loss: 2.3731478536338138
Epoch 14/100, Loss: 2.2565428640011858
Epoch 15/100, Loss: 2.143849967715137
Epoch 16/100, Loss: 2.049211469509249
Epoch 17/100, Loss: 1.952453485407626
Epoch 18/100, Loss: 1.8709452301637273
Epoch 19/100, Loss: 1.7895474732669074
Epoch 20/100, Loss: 1.7197626165877609
Epoch 21/100, Loss: 1.6481014887491863
Epoch 22/100, Loss: 1.5839986119951521
Epoch 23/100, Loss: 1.5284198019140047
Epoch 24/100, Loss: 1.4733692056553107
Epoch 25/100, Loss: 1.4287460840734325
Epoch 26/100, Loss: 1.3779278385609313
Epoch 

In [19]:
del dataset,dataloader,criterion,optimizer,num_epochs

In [20]:
torch.save(model.state_dict(), 'lstm_text_gen.pth')

In [15]:
# Colab drive mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
# copy the model to drive
# !cp lstm_text_gen.pth /content/drive/MyDrive/lstm_text_gen.pth

In [50]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [51]:
# load model state_dict
model = LSTMTextGenerator(vocab_size, embedding_dim, hidden_dim).to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/lstm_text_gen.pth'))

<All keys matched successfully>

In [52]:
def predict_next_word(model, input_sequence, word2idx, idx2word, device):
    model.eval()
    with torch.no_grad():
        input_ids = torch.zeros((1, n_words, len(word2idx)), dtype=torch.float32).to(device)
        for t, word in enumerate(input_sequence):
            if word in word2idx:
                input_ids[0, t, word2idx[word]] = 1
        output = model(input_ids)
        last_output = output[:, -1, :]
        predicted_idx = last_output.argmax(dim=1).item()
        return idx2word[predicted_idx]

In [63]:
def predict_next_words(model, input_sequence, word2idx, idx2word, device):
    model.eval()
    with torch.no_grad():
        input_ids = torch.zeros((1, n_words, len(word2idx)), dtype=torch.float32).to(device)
        for t, word in enumerate(input_sequence):
            if word in word2idx:
                input_ids[0, t, word2idx[word]] = 1

        output = model(input_ids)
        last_output = output[:, -1, :].squeeze(0)  # (vocab_size,)

        # probs = torch.softmax(last_output, dim=0)
        k = min(5, last_output.shape[0])
        top_probs, top_indices = torch.topk(last_output, k=k)

        return [(idx2word[int(i)], float(p)) for p, i in zip(top_probs.cpu(), top_indices.cpu())]

In [67]:
test = "حدّثنا مالك عن "

In [68]:
predict_next_words(model, test.split()[-n_words:], word2idx, idx2word, device)

[(np.str_('نافع'), 0.17132917046546936),
 (np.str_('ابن'), 0.1657857596874237),
 (np.str_('يحيى'), 0.11700371652841568),
 (np.str_('هشام'), 0.10780121386051178),
 (np.str_('عبد'), 0.09769809991121292)]

In [66]:
for _ in range(30):
    test += predict_next_word(model, test.split()[-n_words:], word2idx, idx2word, device) + " "
print(test)

حدّثنا مالك عن ابن شهاب عن سعيد بن المسيب أنه قال كنت مع عبد الله بن عمر كان إذا سئل عن الاستطابة فقال أولا يجد أحدكم ثلاثة أحجار . و حدثني عن مالك عن 


أبرز مرويات "حدثنا مالك عن":

مالك عن نافع عن ابن عمر رضي الله عنهما، وهو الذي قال فيه البخاري: «أصحّ الأسانيد: مالك عن نافع عن ابن عمر».

مالك عن ابن شهاب الزهري عن أنس أو عن السائب بن يزيد أو عن غيرهما من الصحابة، فالزهري من كبار شيوخ مالك في الموطأ.

مالك عن يحيى بن سعيد الأنصاري عن الصحابة، ويحيى بن سعيد من أئمة المدينة الكبار الذين يروي عنهم مالك كثيرًا.

مالك عن هشام بن عروة عن أبيه عروة عن عائشة أو غيرها من الصحابة، وهشام من أثبت تلاميذ عروة الذين اعتمدهم مالك.

مالك عن أبي الزناد عبد الله بن ذكوان عن الأعرج عن أبي هريرة، وهو إسناد مدني مشهور تكرر في كتب السنة، ومنها الموطأ.

مالك عن عبد الله بن دينار عن ابن عمر، وعبد الله بن دينار من أكثر من يروي عن ابن عمر في الموطأ وصحيح البخاري.

